# Soul Tower — Analytics Notebook

This notebook is your sandbox for exploring Soul Tower card and hero data. Everything loads from the local JSON cache produced by the data pipeline, plus optional live data from the Flask analytics server.

**How to use this:**

1. Run the setup cell below once. It loads every sheet into a dict of DataFrames called `sheets`.
2. Each section is independent. Skip around. Modify cells freely.
3. When you find a pattern you'll reuse, promote it to `src/analytics/query.py`.

**Tutorial structure:** I'll annotate each section with *what it does*, *why it matters*, and *how to extend it*.

## Setup

Run this cell every time you open the notebook. It imports pandas and loads all the cached sheets.

If you see a `FileNotFoundError`, run the pipeline first: `python main.py --fresh` from the project root.

In [1]:
import sys
from pathlib import Path

# Make src importable from the notebook
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == 'analytics' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.analytics.query import (
    load_all, load_sheet,
    parse_cost, enrich_costs,
    has_ability,
    spells, brutals, rituals,
    stat_summary, heroes_by_origin,
    live_heroes, live_sessions, merged_hero_view,
)

# Load everything once. Reuse `sheets` throughout the notebook.
sheets = load_all()

# Give yourself friendly shortcut variables
heroes   = sheets['hero']
commons  = sheets['common_cards']
legends  = sheets['legendary']
calams   = sheets['calamity']
villains = sheets['villain']

print(f'Loaded: {len(heroes)} heroes, {len(commons)} commons, {len(legends)} legendaries, {len(calams)} calamities, {len(villains)} villains')

ModuleNotFoundError: No module named 'src'

## Section 1: Hero analytics

Heroes have five stats and an origin. These two cells give you a quick balance check.

In [ ]:
# Overall stat distribution across all heroes.
# Spot outliers: anything way above or below the mean is either a design
# statement (intentional) or a balance flag (unintentional).
stat_summary(sheets)

In [ ]:
# Group heroes by Origin and summarize a specific stat.
# Change the second argument to pivot: 'Health', 'Might', 'Speed', 'Luck', 'Arcana'.
#
# Use this to verify design rules like "all Imanis heroes have at least 5 Health."
heroes_by_origin(sheets, 'Health')

In [ ]:
# Show all heroes sorted by total stat budget. Catches under/over-budgeted heroes.
stat_cols = ['Health', 'Might', 'Speed', 'Luck', 'Arcana']
heroes_budget = heroes.copy()
heroes_budget['total'] = heroes_budget[stat_cols].sum(axis=1)
heroes_budget[['Nickname', 'Alignment', 'Origin'] + stat_cols + ['total']].sort_values('total', ascending=False)

## Section 2: Card cost analysis

Card costs can be fixed integers or die expressions like `2d4`. The `enrich_costs` helper parses both and gives you `cost_min`, `cost_max`, `cost_avg`, and `is_die_cost` columns.

This section is especially useful for balancing: you want your average costs per subtype to feel right for the economy, and you want to see which cards have variable costs that could go either way.

In [ ]:
# Enrich every common card with parsed cost columns
commons_enriched = enrich_costs(commons)

# Average cost by type
commons_enriched.groupby('Type')['cost_avg'].agg(['count', 'mean', 'min', 'max']).round(2)

In [ ]:
# Same for legendaries
legends_enriched = enrich_costs(legends)
legends_enriched.groupby('Type')['cost_avg'].agg(['count', 'mean', 'min', 'max']).round(2)

In [ ]:
# Which cards have die-based costs? These are the ones players have to
# commit-then-roll for. Design review: is the risk/reward right?
commons_enriched[commons_enriched['is_die_cost']][['Name', 'Type', 'Subtype', 'Cost', 'cost_min', 'cost_max', 'cost_avg']]

In [ ]:
# Cost distribution by subtype — do Instants cost what you expect?
# Do Entropies lean high since Foe gains Mana as the cost?
commons_enriched.groupby(['Type', 'Subtype'])['cost_avg'].agg(['count', 'mean']).round(2)

## Section 3: Ability search

The `has_ability` function searches every effect and passive field across every sheet. Use it to see where a specific mechanic shows up in the game.

This is the notebook's killer feature. Want to see everything touching Enchant? One call.

In [ ]:
# Change this to any ability name: Enchant, Fortune, Blood Price, Harness, etc.
ABILITY = 'Enchant'

results = has_ability(ABILITY, sheets)
print(f'Found {len(results)} mentions of {ABILITY!r}')
print()
print('By source:')
print(results.groupby('_source').size())

In [ ]:
# View the actual matches. Adjust which columns to show for readability.
display_cols = ['_source', 'Name', 'Nickname', 'Type', 'Subtype', 'Cost', 'Effect', 'Passive Effect', '_matched_in']
available = [c for c in display_cols if c in results.columns]
results[available]

In [ ]:
# Break down an ability by card type
# Useful for checking mechanical coverage: is Fortune well-represented across
# all three card types, or only on Spells?
if 'Type' in results.columns:
    print(results.groupby('Type').size())

## Section 4: Type and subtype filters

Quick accessors for looking at just Spells, just Brutals, or just Rituals. Combines commons and legendaries.

In [ ]:
# Every Spell in the game
all_spells = spells(sheets)
print(f'{len(all_spells)} Spells total')
all_spells[['Name', 'Subtype', 'Cost', '_source']].head(20)

In [ ]:
# Every Brutal. Which subtypes are most common?
all_brutals = brutals(sheets)
all_brutals.groupby('Subtype').size()

In [ ]:
# Every Ritual with an Entropy subtype (the ones your Foe pays for).
# Worth watching closely since they shift Mana to the other side.
entropy_rituals = rituals(sheets)
entropy_rituals = entropy_rituals[entropy_rituals['Subtype'] == 'Entropy']
entropy_rituals[['Name', 'Cost', 'Effect', '_source']]

## Section 5: Cursed cards

Cursed cards cannot be exiled and inflict Pain. How many are in the game, and are they appropriately distributed across types?

In [ ]:
# Cursed cards in the common pool
cursed_commons = commons[commons['Cursed'] == True]
print(f'{len(cursed_commons)} Cursed commons ({len(cursed_commons) / len(commons):.0%} of pool)')
cursed_commons[['Name', 'Type', 'Subtype', 'Cost', 'Effect']]

In [ ]:
# Cursed legendaries
cursed_legends = legends[legends['Cursed'] == True]
print(f'{len(cursed_legends)} Cursed legendaries')
cursed_legends[['Name', 'Hero', 'Type', 'Subtype', 'Cost', 'Effect']]

## Section 6: Hero-legendary linkage

Each hero has two legendary cards. This section verifies the linkage and explores whether legendaries align with their hero's identity.

For example, if a hero's passive is about Fortune, do their legendaries use Fortune?

In [ ]:
# Count legendaries per hero — should always be 2
legend_counts = legends.groupby('Hero').size().rename('legendary_count').reset_index()
missing = legend_counts[legend_counts['legendary_count'] != 2]
if len(missing) > 0:
    print(f'WARNING: {len(missing)} heroes have != 2 legendaries:')
    print(missing)
else:
    print('All heroes have exactly 2 legendaries ✓')
legend_counts

In [ ]:
# Show each hero with their passive and their two legendaries side by side
hero_view = heroes[['Nickname', 'Alignment', 'Origin', 'Passive Effect']].copy()

legend_pairs = (
    legends.groupby('Hero')
    .agg(legendaries=('Name', lambda s: ' + '.join(s)))
    .reset_index()
)

hero_view.merge(legend_pairs, left_on='Nickname', right_on='Hero', how='left')

## Section 7: Live analytics (Flask)

These cells query your running Flask analytics server. They will fail silently if the server is not running, returning empty DataFrames.

Start the server first: `python backend/analytics_server.py`

In [ ]:
# Pull playtest session list
sessions_df = live_sessions()
print(f'{len(sessions_df)} sessions logged')
sessions_df.tail(10)

In [ ]:
# Pull live hero pick/defeat stats
live = live_heroes()
live

In [ ]:
# Merge static design data with live playtest stats.
# Heroes with pick_count == 0 have never been picked. That's a flag.
# Heroes with a high pick_count AND high defeat_count are the hotspots.
merged = merged_hero_view(sheets)
merged.sort_values('pick_count', ascending=False)[['Nickname', 'Alignment', 'Health', 'Might', 'pick_count', 'defeat_count', 'avg_defeat_turn']]

## Section 8: Extending the notebook

When you discover a pattern worth keeping, here's the path:

1. **Draft it in the notebook.** Get it working with the data in front of you.
2. **Generalize it.** Replace hardcoded sheet names or column names with parameters.
3. **Move it to `src/analytics/query.py`.** Add a docstring. Add it to the imports at the top of this notebook.
4. **Reload.** Use `importlib.reload` or restart the kernel.

Example: Let's say you find yourself computing card-cost distributions a lot. Below is a one-off version. When you use it three times, promote it to `query.py` as `cost_distribution(card_type)`.

In [ ]:
# One-off example: count cards at each cost for each type
commons_enriched.groupby(['Type', 'cost_avg']).size().unstack(fill_value=0)

In [ ]:
# Your scratch space below. Add cells for whatever you want to explore next.
